# Analisis hasil E14 sebelum E15

Klik **Run all**, lalu unggah **`Hasil_AES_Reset_E14_S1_S2_S3_MultiTrait.zip`**
dari output E14. Kalau sudah diekstrak, unggah `04_Prediksi_dan_Evaluasi.xlsx` saja.
GPU tidak diperlukan.

Kode mengenali skenario dari metadata di workbook 04. Pada ekspor E14, ZIP
lengkap berisi evaluasi **S3**. Data S1 dan S2 bisa ditambahkan melalui masing-masing
ZIP ringkasnya atau workbook 04 yang diambil dari ZIP tersebut. Kalau ada keduanya,
notebook akan membandingkan S1 dan S2 pada lima fold yang sama.

[Lihat notebook training E14](https://colab.research.google.com/github/MalikHariyanto/Automated_Essay_Scoring/blob/codex/aes-learning/eksperimen/03_tiga_input_multi_trait/notebooks/AES_Reset_E14_S1_S2_S3_MultiTrait.ipynb).


In [ ]:
from io import BytesIO
import json
from pathlib import PurePosixPath
from zipfile import ZipFile

from IPython.display import display
import numpy as np
import pandas as pd

SCENARIOS = {'S1': 'S1 RoBERTa', 'S2': 'S2 BGE-M3', 'S3': 'S3 Gabungan'}
TRAITS = ('isi', 'organisasi', 'kebahasaan', 'mekanik')
METRICS = ('RMSE', 'Pearson', 'QWK', 'Gwet AC2')
KEYS = ['id_jawaban', 'fold', 'kelompok', 'no_soal', 'baris_excel',
        *(f'{trait}_label' for trait in TRAITS)]


def evaluation_workbooks(uploads):
    """Ambil workbook 04 dari ZIP atau dari file XLSX yang sudah diekstrak."""
    for filename, contents in uploads.items():
        name = PurePosixPath(filename).name.lower()
        if name.endswith('.zip'):
            with ZipFile(BytesIO(contents)) as archive:
                members = [entry for entry in archive.infolist()
                           if PurePosixPath(entry.filename).name == '04_Prediksi_dan_Evaluasi.xlsx']
                if not members:
                    raise ValueError(f'{filename} tidak menyediakan workbook 04.')
                for member in members:
                    yield f'{filename}/{member.filename}', archive.read(member)
        elif name.startswith('04_prediksi_dan_evaluasi') and name.endswith('.xlsx'):
            yield filename, contents


def load_oof(filename, contents):
    with pd.ExcelFile(BytesIO(contents), engine='openpyxl') as excel:
        info = excel.parse('Info')
        metadata = dict(zip(info['field'], info['value']))
        config = json.loads(metadata['final_config'])
        scenario = config.get('scenario')
        if config.get('experiment') != 'Reset_E14' or scenario not in SCENARIOS:
            raise ValueError(f'{filename}: gunakan workbook evaluasi E14 untuk S1, S2, atau S3.')
        predictions = excel.parse('Prediksi OOF').sort_values('id_jawaban').reset_index(drop=True)
        oof = excel.parse('CV OOF per Trait').set_index('trait').loc[list(TRAITS)]
        test = excel.parse('Test per Trait').set_index('trait').loc[list(TRAITS)]
        summary = excel.parse('Ringkasan CV').iloc[0]

    if (predictions[KEYS].isna().any().any()
        or predictions['id_jawaban'].duplicated().any()
        or len(predictions) != int(metadata['development_rows'])
        or set(predictions['fold']) != {1, 2, 3, 4, 5}):
        raise ValueError(f'Prediksi validasi {scenario} tidak lengkap.')
    for trait in TRAITS:
        residual = predictions[f'{trait}_prediksi_mentah'] - predictions[f'{trait}_label']
        rmse = np.sqrt(residual.pow(2).mean())
        if not np.isfinite(residual).all() or not np.isclose(rmse, oof.loc[trait, 'RMSE'], atol=1e-10, rtol=0):
            raise ValueError(f'Prediksi dan rekap RMSE {scenario} berbeda pada trait {trait}.')
    return scenario, metadata, predictions, oof, test, summary


def load_runs(uploads):
    runs = {}
    for filename, contents in evaluation_workbooks(uploads):
        scenario, *run = load_oof(filename, contents)
        if scenario in runs:
            raise ValueError(f'Ada dua workbook 04 untuk {scenario}. Unggah satu per skenario.')
        runs[scenario] = run
    if not runs:
        raise ValueError('Unggah ZIP E14 atau file 04_Prediksi_dan_Evaluasi.xlsx.')
    return {scenario: runs[scenario] for scenario in SCENARIOS if scenario in runs}


def describe_run(predictions):
    folds, low_scores = [], []
    for trait in TRAITS:
        label = predictions[f'{trait}_label']
        residual = predictions[f'{trait}_prediksi_mentah'] - label
        category = predictions[f'{trait}_prediksi_kategori']
        for fold, errors in residual.groupby(predictions['fold']):
            folds.append({'trait': trait, 'fold': fold, 'RMSE': np.sqrt(errors.pow(2).mean())})
        low_scores.append({'trait': trait,
                           'jumlah label 1': int(label.eq(1).sum()),
                           'label 1 terdeteksi': int((label.eq(1) & category.eq(1)).sum()),
                           'label 2 keliru jadi 1': int((label.eq(2) & category.eq(1)).sum())})
    return pd.DataFrame(folds), pd.DataFrame(low_scores)


def compare_s1_s2(runs):
    info1, s1, m1, _, _ = runs['S1']
    info2, s2, m2, _, _ = runs['S2']
    for key in ('source_sha256', 'split_sha256'):
        if not info1.get(key) or info1[key] != info2.get(key):
            raise ValueError('Data atau pembagian fold S1 dan S2 berbeda.')
    if not s1[KEYS].equals(s2[KEYS]):
        raise ValueError('Baris jawaban, label, atau fold S1 dan S2 berbeda.')
    comparison = pd.concat({SCENARIOS['S1']: m1[list(METRICS)],
                            SCENARIOS['S2']: m2[list(METRICS)]}, axis=1)
    first = describe_run(s1)[0].rename(columns={'RMSE': 'RMSE S1'})
    second = describe_run(s2)[0].rename(columns={'RMSE': 'RMSE S2'})
    folds = first.merge(second, on=['trait', 'fold'], validate='one_to_one')
    folds['selisih S2 − S1'] = folds['RMSE S2'] - folds['RMSE S1']
    return comparison, folds


def show_results(runs):
    print('Skenario tersedia:', ', '.join(runs))
    for scenario, (_, predictions, oof, test, summary) in runs.items():
        print(f"\n{SCENARIOS[scenario]} · RMSE rata-rata 5 fold {summary['mean_cv_RMSE']:.4f}"
              f" · refit {int(summary['epoch_refit'])} epoch")
        display(pd.concat({'Validasi (OOF)': oof[list(METRICS)],
                           'Test': test[list(METRICS)]}, axis=1).round(4))
        folds, low_scores = describe_run(predictions)
        if scenario == 'S3':
            print('RMSE validasi per fold')
            display(folds.pivot(index='fold', columns='trait', values='RMSE').loc[:, list(TRAITS)].round(4))
        print('Deteksi skor 1 di validasi')
        display(low_scores)
    if {'S1', 'S2'} <= runs.keys():
        first, second = runs['S1'][0], runs['S2'][0]
        if first.get('notebook_code_sha256') != second.get('notebook_code_sha256'):
            print('S1 dan S2 berasal dari versi notebook berbeda. Cek konfigurasi training sebelum menarik kesimpulan.')
        comparison, folds = compare_s1_s2(runs)
        print('Empat metrik validasi S1 dan S2')
        display(comparison.round(4))
        print('RMSE per fold. Selisih positif berarti RMSE S1 lebih rendah.')
        display(folds.round(4))
    else:
        missing = ', '.join(s for s in ('S1', 'S2') if s not in runs)
        print(f'Untuk membandingkan kedua encoder, tambahkan hasil {missing}.')


## Unggah hasil E14

Pilih ZIP lengkap, ZIP ringkas, atau workbook 04. Kalau lo memilih seluruh
folder yang sudah diekstrak, kode mengambil workbook 04 dari daftar file untuk
analisis. Untuk menambah hasil lain, unggah workbook 04 yang belum diunggah
atau ZIP ringkasnya dalam satu kali pemilihan bersama file hasil sebelumnya.


In [ ]:
from google.colab import files

uploaded = files.upload()
runs = load_runs(uploaded)
show_results(runs)


## Buat keputusan E15 dari validasi

RMSE yang lebih kecil berarti prediksi lebih dekat ke skor guru. Cek apakah S1
dan S2 unggul pada aspek yang berbeda dan apakah pola itu muncul di beberapa fold.
Tabel test membantu membaca hasil run; gunakan validasi untuk memilih perubahan S3.
